# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa — Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The dataset is provided in the [MLCommons Croissant](https://mlcommons.org/croissant/) format, which allows for rich, machine-readable metadata and seamless programmatic data access.

### Dataset Source
- **Croissant schema URL:** [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

In [ ]:
# Install the mlcroissant library if not already installed
!pip install mlcroissant --quiet

## 1. Data Loading

Load the dataset metadata and records using `mlcroissant`. The Croissant schema provides both data and a description of its structure including record sets, fields, types, and distribution.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Explore available record sets, their field `@id`s, and columns. The Croissant metadata defines all data assets in the dataset. We’ll look for entities of type `RecordSet` and enumerate their fields and columns.

In [ ]:
# List all record sets, fields, and columns with their @id

record_sets = [rs for rs in metadata.record_sets]
for rs in record_sets:
    print(f"[RecordSet] name: {rs.name}, @id: {rs.id}")
    print(f"  Description: {rs.description}")
    if hasattr(rs, 'fields') and rs.fields:
        for field in rs.fields:
            print(f"    [Field] {field.name} – @id: {field.id} (type: {field.data_type})")
            if hasattr(field, 'columns') and field.columns:
                for col in field.columns:
                    print(f"      [Column] {col.name} – @id: {col.id} (type: {col.data_type})")
    print()

## 3. Data Extraction

Load records from one or more record sets into Pandas DataFrames for analysis. Use the `@id` of a record set and fields you wish to extract, as shown in the previous step.

In [ ]:
# If there are multiple record sets, select all by their @id. List will be built dynamically.
record_set_ids = [rs.id for rs in record_sets]
dfs = {}

for rs_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {rs_id}")
    records_iter = dataset.records(record_set=rs_id)
    df = pd.DataFrame(list(records_iter))
    print(f"Columns: {df.columns.tolist()}")
    print(df.head())
    dfs[rs_id] = df

## 4. Exploratory Data Analysis (EDA)

Let’s process the main survey DataFrame (the largest RecordSet, typically containing individual-level survey data), performing filtering and basic transformation steps. We'll use the `@id` of fields from above. You can adjust the selected fields depending on the actual fields discovered.

In [ ]:
# Select main dataset (assume the first record set is the main table; adjust if not correct)
main_rs_id = record_set_ids[0]
main_df = dfs[main_rs_id]

# Show basic field names
print(f"Available fields in main RecordSet ({main_rs_id}):")
print(main_df.columns.tolist())

# Pick a numeric field for demonstration; replace with appropriate @id as displayed above
# For this dataset, GAD-7 or PHQ-9 scores are good numeric fields (e.g., 'gad7_score', 'phq9_score', '@id:field_GAD7_Total')
# If not known, pick one field below; adjust as needed
candidate_numeric_fields = [col for col in main_df.columns if 'score' in col.lower() or 'gad7' in col.lower() or 'phq9' in col.lower()]
if candidate_numeric_fields:
    numeric_field = candidate_numeric_fields[0]
    print(f"Using numeric field: {numeric_field}")
else:
    numeric_field = main_df.columns[0]  # fallback

# Filter records with score above a threshold
threshold = 10
filtered_df = main_df[main_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head(3))

# Normalize the numeric score
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} (first rows):")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a demographic field (replace with appropriate @id revealed above, e.g., 'level_of_education', 'gender', etc.)
candidate_group_fields = [col for col in main_df.columns if any(x in col.lower() for x in ['gender', 'education', 'marital'])]
if candidate_group_fields:
    group_field = candidate_group_fields[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nMean {numeric_field} grouped by {group_field}:")
    print(grouped_df.head())

## 5. Visualization

Visualize the distribution of selected numeric scores, and (if available) compare scores by a demographic attribute.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot for the selected numeric field
plt.figure(figsize=(8,5))
sns.histplot(main_df[numeric_field].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# If group_field exists, plot boxplot
if 'group_field' in locals():
    plt.figure(figsize=(8,5))
    sns.boxplot(x=main_df[group_field], y=main_df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

- The FAIR² Mental Health Survey Dataset from Kilifi County enables streamlined, reproducible data analysis via its Croissant schema and the `mlcroissant` Python library.
- We've demonstrated how to dynamically explore record sets, load them with their semantic `@id`, and conduct elementary filtering, normalization, and grouping for EDA.
- Visualizations reveal basic data quality and value distributions, supporting further domain-specific studies.

**Next steps:** Consult the field descriptions and Croissant metadata for further specialized analyses, and ensure all sensitive columns are handled according to ethical guidelines.